# Load Data

In [1]:
import json
import mlflow
import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType

from utils import get_table

In [2]:
# Utils
def time_based_split(df: pd.DataFrame, time_col: str, train_frac=0.7, val_frac=0.15):
    """
    Split per tempo: train (più vecchio), val (intermedio), test (più recente).
    """
    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    n = len(df_sorted)

    n_train = int(n * train_frac)
    n_val = int(n * val_frac)

    train_df = df_sorted.iloc[:n_train]
    val_df = df_sorted.iloc[n_train:n_train + n_val]
    test_df = df_sorted.iloc[n_train + n_val:]
    return train_df, val_df, test_df


def infer_column_types(df: pd.DataFrame, target_col: str):
    """
    Infer column types from a pandas DataFrame.

    Returns
    -------
    num_cols : list
        Numeric columns (int, float)
    bool_cols : list
        Boolean columns
    datetime_cols : list
        Datetime columns
    cat_cols : list
        Categorical / object columns
    """
    df = df.drop(columns=[target_col])

    num_cols = []
    bool_cols = []
    datetime_cols = []
    cat_cols = []

    for col in df.columns:
        dtype = df[col].dtype

        if pd.api.types.is_bool_dtype(dtype):
            bool_cols.append(col)

        elif pd.api.types.is_numeric_dtype(dtype):
            num_cols.append(col)

        elif pd.api.types.is_datetime64_any_dtype(dtype):
            datetime_cols.append(col)

        elif pd.api.types.is_categorical_dtype(dtype) or pd.api.types.is_object_dtype(dtype):
            cat_cols.append(col)

        else:
            # fallback (rare types)
            cat_cols.append(col)

    return num_cols, bool_cols, datetime_cols, cat_cols


def bool_to_int(X: pd.DataFrame):
    Xc = X.copy()
    for col in Xc.columns:
        Xc[col] = Xc[col].astype("int8")
    return Xc


def datetime_to_int64(X: pd.DataFrame):
    """
    Converte datetime -> int64 (nanosecondi dal 1970). Evita feature engineering extra.
    """
    Xc = X.copy()
    for col in Xc.columns:
        # pandas datetime64[ns]
        Xc[col] = pd.to_datetime(Xc[col], errors="coerce").astype("int64")
    return Xc

def get_feature_names_from_preprocessor(preprocessor: ColumnTransformer):
    """
    Estrae i nomi feature dopo ColumnTransformer (incluse one-hot).
    """
    output_features = []
    for name, trans, cols in preprocessor.transformers_:
        if name == "remainder" and trans == "drop":
            continue
        if trans == "passthrough":
            # cols è lista di colonne
            output_features.extend(list(cols))
        else:
            # pipeline o transformer singolo
            if isinstance(trans, Pipeline):
                last = trans.steps[-1][1]
            else:
                last = trans

            if hasattr(last, "get_feature_names_out"):
                # OneHotEncoder / ecc.
                fn = last.get_feature_names_out(cols)
                output_features.extend(list(fn))
            else:
                # fallback: usa i nomi originali
                output_features.extend(list(cols))
    return output_features

In [3]:
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Users/maicolnicolini96@gmail.com/bet_analytics_experiments/lightgbm_training")

target_col = "win_1"
time_col = "time"
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """
df_loaded = get_table(query)
df_loaded[target_col] = df_loaded[target_col] == "true"
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

# Data Processing

## Train/Test/Val Split

In [4]:
# experiment_name: str = "lgbm_binary_time_split",
# run_name: str = "baseline",
params=None
run_name = "test"
train_frac: float = 0.8
val_frac: float = 0.1
drop_importance_below: float = 0.0  # es. 0.0 = niente drop, oppure 1e-6 / 0.0001
onnx_export_path: str = "artifacts/model.onnx"

unuseful_cols = ['team_league', 'team_home', 'team_away']

df = df_loaded.copy()

df = df.drop(unuseful_cols, axis=1)
# Parse time col
df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

# Basic sanity
df = df.dropna(how='all', axis=1)
# target must be 0/1
df[target_col] = df[target_col].astype(int)

num_cols, bool_cols, datetime_cols, cat_cols = infer_column_types(df, target_col)

df["white_noise"] = np.random.normal(
    loc=df[num_cols].mean().mean(),
    scale=df[num_cols].std().mean(),
    size=len(df[num_cols])
)

# num_cols.append("white_noise")
train_df, val_df, test_df = time_based_split(df, time_col, train_frac, val_frac)


X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col].values

X_val = val_df.drop(columns=[target_col])
y_val = val_df[target_col].values

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col].values

# Preprocess
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("astype", FunctionTransformer(lambda x: x.astype("float64"), validate=False)),
            ]), num_cols),
        ("bool", Pipeline(steps=[
            ("to_df", FunctionTransformer(lambda x: pd.DataFrame(x, columns=bool_cols), validate=False)),
            ("cast", FunctionTransformer(bool_to_int, validate=False)),
        ]), bool_cols),
        ("dt", Pipeline(steps=[
            ("to_df", FunctionTransformer(lambda x: pd.DataFrame(x, columns=datetime_cols), validate=False)),
            ("cast", FunctionTransformer(datetime_to_int64, validate=False)),
        ]), datetime_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
    verbose_feature_names_out=False,
)

# Default LGBM params (puoi modificarli)
lgbm_params = {
    "n_estimators": 2000,
    "learning_rate": 0.03,
    "num_leaves": 64,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "random_state": 42,
    "n_jobs": -1,
    "objective": "binary",
    "metric": "auc",
}

if params:
    lgbm_params.update(params)

model = LGBMClassifier(**lgbm_params)

# pipeline sklearn
pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", model),
])


In [6]:
with mlflow.start_run(run_name=run_name) as mlflow_run:
    # Log split info
    mlflow.log_params({
        "train_frac": train_frac,
        "val_frac": val_frac,
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        **{f"lgbm__{k}": v for k, v in lgbm_params.items()},
    })

    # Fit with early stopping using validation
    # NB: early_stopping via fit params (LightGBM sklearn API)
    pipe.fit(
        X_train, y_train,
        clf__eval_set=[(preprocessor.fit_transform(X_val), y_val)],  # val transformed
        clf__eval_metric="auc",
        clf__callbacks=[],
    )

    # Predict proba
    p_train = pipe.predict_proba(X_train)[:, 1]
    p_val = pipe.predict_proba(X_val)[:, 1]
    p_test = pipe.predict_proba(X_test)[:, 1]

    auc_train = roc_auc_score(y_train, p_train) if len(np.unique(y_train)) > 1 else np.nan
    auc_val = roc_auc_score(y_val, p_val) if len(np.unique(y_val)) > 1 else np.nan
    auc_test = roc_auc_score(y_test, p_test) if len(np.unique(y_test)) > 1 else np.nan

    mlflow.log_metrics({
        "auc_train": float(auc_train) if np.isfinite(auc_train) else -1.0,
        "auc_val": float(auc_val) if np.isfinite(auc_val) else -1.0,
        "auc_test": float(auc_test) if np.isfinite(auc_test) else -1.0,
    })

    # Feature importance
    prep_fitted = pipe.named_steps["prep"]
    feature_names = get_feature_names_from_preprocessor(prep_fitted)

    booster = pipe.named_steps["clf"].booster_
    importances = booster.feature_importance(importance_type="gain")
    imp_df = pd.DataFrame({
        "feature": feature_names,
        "importance_gain": importances
    }).sort_values("importance_gain", ascending=False)

    imp_csv = "feature_importance_gain.csv"
    imp_df.to_csv(imp_csv, index=False)
    mlflow.log_artifact(imp_csv)

    # drop_importance_below = float(imp_df.set_index(["feature"], drop=True).loc["white_noise"].values)
    drop_importance_below = float(imp_df.set_index(["feature"], drop=True).iloc[9].values)


    # Optional drop features under threshold & retrain
    if drop_importance_below > 0.0:
        keep_mask = imp_df["importance_gain"].values > drop_importance_below
        kept_features = imp_df.loc[keep_mask, "feature"].tolist()
        dropped = int((~keep_mask).sum())
        mlflow.log_params({
            "drop_importance_below": drop_importance_below,
            "dropped_features_count": dropped,
            "kept_features_count": len(kept_features),
        })

        # Per droppare in modo robusto con one-hot: selezioniamo colonne DOPO il preprocessor
        # Strategy: trasformiamo X_* e poi addestriamo un secondo LGBM su matrice ridotta.
        Xtr = prep_fitted.transform(X_train)
        Xva = prep_fitted.transform(X_val)
        Xte = prep_fitted.transform(X_test)

        keep_idx = np.where(keep_mask)[0]
        Xtr_k = Xtr[:, keep_idx]
        Xva_k = Xva[:, keep_idx]
        Xte_k = Xte[:, keep_idx]

        model2 = LGBMClassifier(**lgbm_params)
        model2.fit(
            Xtr_k, y_train,
            eval_set=[(Xva_k, y_val)],
            eval_metric="auc",
        )

        p_val2 = model2.predict_proba(Xva_k)[:, 1]
        p_test2 = model2.predict_proba(Xte_k)[:, 1]
        auc_val2 = roc_auc_score(y_val, p_val2) if len(np.unique(y_val)) > 1 else np.nan
        auc_test2 = roc_auc_score(y_test, p_test2) if len(np.unique(y_test)) > 1 else np.nan

        mlflow.log_metrics({
            "auc_val_dropped": float(auc_val2) if np.isfinite(auc_val2) else -1.0,
            "auc_test_dropped": float(auc_test2) if np.isfinite(auc_test2) else -1.0,
        })

        # Log modello ridotto come artifact “secondario”
        mlflow.lightgbm.log_model(model2, artifact_path="lgbm_model_retrained_after_drop")
        # Salviamo anche gli indici keep per riprodurre a runtime
        with open("artifacts/kept_feature_indices.json", "w") as f:
            json.dump(keep_idx.tolist(), f)
        mlflow.log_artifact("artifacts/kept_feature_indices.json")

    # Log modello pipeline (preprocess + lgbm)
    mlflow.sklearn.log_model(pipe, artifact_path="sklearn_pipeline_lgbm")

    # ------------------------------------
    # ONNX export (modello puro LightGBM)
    # ------------------------------------
    # Per ONNX più compatto: esportiamo il Booster e a runtime replichi il preprocessing.
    # Qui esportiamo il modello *addestrato sullo spazio trasformato*:
    Xtr_trans = prep_fitted.transform(X_train)
    n_features_trans = Xtr_trans.shape[1]


    # Convert LightGBM booster to ONNX
    # NOTE: output probabilità: dipende dal converter; spesso produce label+probabilities
    initial_types = [("input", FloatTensorType([None, n_features_trans]))]
    onnx_model = onnxmltools.convert_lightgbm(
        booster,
        initial_types=initial_types,
        target_opset=15,
    )

    with open(onnx_export_path, "wb") as f:
        f.write(onnx_model.SerializeToString())

    mlflow.log_artifact(onnx_export_path)

[LightGBM] [Info] Number of positive: 1742, number of negative: 2281
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001141 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 24440
[LightGBM] [Info] Number of data points in the train set: 4023, number of used features: 131
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.433010 -> initscore=-0.269580
[LightGBM] [Info] Start training from score -0.269580
[LightGBM] [Info] Number of positive: 1742, number of negative: 2281
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000250 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2278
[LightGBM] [Info] Number of data points in the train set: 4023, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.433010 -> initscore=-0.269580
[LightGBM] [Info] Start training from score -0.269580


2026/01/28 15:02:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 15:02:37 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/01/28 15:02:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 15:02:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run test at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/123563306875498/runs/28e56dbd7c564af4b5be5ee26d0e82ed
🧪 View experiment at: https://dbc-49215966-9ab1.cloud.databricks.com/ml/experiments/123563306875498


# Calculate Treshold and EV

In [7]:
X_val = val_df.drop(target_col, axis=1)
y_val = val_df[target_col]
X_test = test_df.drop(target_col, axis=1)
y_test = test_df[target_col]

In [13]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import precision_score

odds_name = "chance1x2_quote_current1"

def get_threshold_and_test_ev(
    val_df, test_df,
    model, odds_name, target_col,
    P_MIN=0.75, N_MIN_PERC=0.1,
    EV_MIN=0.1, EV_MAX=0.5,
    weight_ev=0.7, weight_nbets=0.3
):
    assert abs(weight_ev + weight_nbets - 1.0) < 1e-6

    X_val = val_df.drop(target_col, axis=1)
    y_val = val_df[target_col]
    X_test = test_df.drop(target_col, axis=1)
    y_test = test_df[target_col]



    # -----------------------------
    # 1. Load model
    # -----------------------------

    # -----------------------------
    # 2. Calibration (fit on VAL)
    # -----------------------------
    calibrator = CalibratedClassifierCV(
        estimator=model,
        method="isotonic",
        cv="prefit"
    )
    calibrator.fit(X_val, y_val)

    # -----------------------------
    # 3. Predict on VAL
    # -----------------------------
    p_val = calibrator.predict_proba(X_val)[:, 1]


    val_df["p_cal"] = p_val
    val_df["odds"] = val_df[odds_name]
    val_df["EV"] = val_df["p_cal"] * val_df["odds"] - 1

    # -----------------------------
    # 4. Sweep thresholds (VAL)
    # -----------------------------
    results = []
    THRESHOLDS = np.linspace(0.55, 0.90, 71)
    N_MIN = int(len(y_val) * N_MIN_PERC)

    for t in THRESHOLDS:
        sel = val_df[val_df["p_cal"] >= t]
        if len(sel) < N_MIN:
            continue

        precision = precision_score(
            sel[target_col].astype(int),
            np.ones(len(sel))
        )
        mean_ev = sel["EV"].mean()

        if precision < P_MIN or not (EV_MIN <= mean_ev <= EV_MAX):
            continue

        results.append({
            "threshold": t,
            "mean_EV_val": mean_ev,
            "precision_val": precision,
            "n_bets_val": len(sel),
            "n_bets_val_perc": int(len(sel) / len(y_val) * 100),
        })

    res_df = pd.DataFrame(results)
    if res_df.empty:
        print("Nessuna soglia valida trovata")
        return None

    # -----------------------------
    # 5. Score robusto (VAL)
    # -----------------------------
    EV_CENTER = (EV_MIN + EV_MAX) / 2

    res_df["score"] = (
        weight_nbets * (res_df["n_bets_val"] / res_df["n_bets_val"].max()) +
        weight_ev * (
            1 - abs(res_df["mean_EV_val"] - EV_CENTER) / (EV_MAX - EV_MIN)
        )
    )

    optimal = res_df.sort_values("score", ascending=False).iloc[0]
    t_star = optimal["threshold"]

    # -----------------------------
    # 6. FINAL EVALUATION ON TEST
    # -----------------------------
    p_test = calibrator.predict_proba(X_test)[:, 1]

    test_df["p_cal"] = p_test
    test_df["odds"] = test_df[odds_name]
    test_df["EV"] = test_df["p_cal"] * test_df["odds"] - 1

    sel_test = test_df[test_df["p_cal"] >= t_star]

    test_metrics = {
        "threshold": t_star,
        "n_bets_test_perc": int(len(sel_test) / len(y_test) * 100),
        "mean_EV_test": sel_test["EV"].mean() if len(sel_test) > 0 else np.nan,
        "ROI_test": sel_test["EV"].sum() / len(sel_test) if len(sel_test) > 0 else np.nan,
        "precision_test": precision_score(
            sel_test[target_col].astype(int),
            np.ones(len(sel_test))
        ) if len(sel_test) > 0 else np.nan
    }

    return {
        "val": optimal.to_dict(),
        "test": test_metrics
    }


optimal = get_threshold_and_test_ev(
    val_df, test_df, pipe, odds_name, target_col,
    P_MIN=0.75, EV_MIN=0.05, EV_MAX=0.5,
    weight_ev=0.8, weight_nbets=0.2, N_MIN_PERC=0.01
)
print(optimal)

{'val': {'threshold': 0.73, 'mean_EV_val': 0.24117647058823527, 'precision_val': 1.0, 'n_bets_val': 17.0, 'n_bets_val_perc': 3.0, 'score': 0.7884407096171803}, 'test': {'threshold': np.float64(0.73), 'n_bets_test_perc': 2, 'mean_EV_test': np.float64(0.25069136372705864), 'ROI_test': np.float64(0.25069136372705864), 'precision_test': 0.75}}


In [9]:
X_test.groupby(df["time"].dt.to_period("W")).count()["time"].median()

np.float64(152.0)

In [87]:
import numpy as np
import pandas as pd
from sklearn.utils import resample

def run_respiro_analysis(
    val_df,
    test_df,
    model,
    odds_name,
    target_col,
    n_runs=50,
    bootstrap_val=True,
    random_state=42,
    **kwargs
):
    """
    Esegue chiamate ripetute a get_threshold_and_test_ev
    e restituisce un DataFrame di robustezza
    """

    rng = np.random.RandomState(random_state)
    rows = []

    for i in range(n_runs):
        # -----------------------------
        # Bootstrap o shuffle su VAL
        # -----------------------------
        if bootstrap_val:
            val_sample = resample(
                val_df,
                replace=True,
                n_samples=len(val_df),
                random_state=rng.randint(0, 10_000)
            )
        else:
            val_sample = val_df.sample(
                frac=1,
                random_state=rng.randint(0, 10_000)
            )

        out = get_threshold_and_test_ev(
            val_sample,
            test_df,
            model,
            odds_name,
            target_col,
            **kwargs
        )

        if out is None:
            continue

        rows.append({
            "run": i,
            # ---- VAL
            "threshold": out["val"]["threshold"],
            "mean_EV_val": out["val"]["mean_EV_val"],
            "precision_val": out["val"]["precision_val"],
            "n_bets_val_perc": out["val"]["n_bets_val_perc"],
            "score": out["val"]["score"],
            # ---- TEST
            "mean_EV_test": out["test"]["mean_EV_test"],
            "precision_test": out["test"]["precision_test"],
            "n_bets_test_perc": out["test"]["n_bets_test_perc"],
            "ROI_test": out["test"]["ROI_test"],
        })

    respiro_df = pd.DataFrame(rows)
    return respiro_df

respiro_df = run_respiro_analysis(
    val_df,
    test_df,
    pipe,
    odds_name,
    target_col,
    n_runs=100,
    bootstrap_val=True,
    P_MIN=0.75,
    EV_MIN=0.05,
    EV_MAX=0.5,
    weight_ev=0.5,
    weight_nbets=0.5,
    N_MIN_PERC=0.01
)


In [96]:
df_loaded.sort_values(by='time', ascending=True)

,time,chance1x2_chance_p1,chance1x2_chance_px,chance1x2_chance_p2,chance1x2_chance_p1x,chance1x2_chance_p2x,chance1x2_chance_p12,chance1x2_chance_pHt1,chance1x2_chance_pHtx,chance1x2_chance_pHt2,...,underOver_flashback_under15,underOver_flashback_over15,underOver_flashback_under25,underOver_flashback_over25,underOver_flashback_under35,underOver_flashback_over35,team_goal,team_goalHt,team_corner,win_1
0,2025-01-10 00:00:00+00:00,29.1,26.9,44.0,56.0,70.9,73.1,27.7,43.9,28.4,...,35.7,64.3,62.4,37.6,82.3,17.7,None,None,None,False
1,2025-01-10 02:30:00+00:00,53.8,26.0,20.2,79.8,46.2,74.0,43.9,40.8,15.3,...,33.6,66.4,59.0,41.0,79.5,20.5,None,None,None,False
2,2025-01-10 02:30:00+00:00,42.2,31.1,26.7,73.3,57.8,68.9,30.8,39.5,29.7,...,23.8,76.2,46.2,53.8,71.0,29.0,None,None,None,True
3,2025-01-10 02:35:00+00:00,37.9,33.6,28.5,71.5,62.1,66.4,24.4,52.2,23.4,...,35.4,64.6,62.1,37.9,83.3,16.7,None,None,None,False
4,2025-01-10 18:00:00+00:00,27.2,34.0,38.8,61.2,72.8,66.0,27.1,37.6,35.3,...,23.8,76.2,46.3,53.7,69.3,30.7,None,None,None,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5023,2026-12-01 18:30:00+00:00,46.9,24.5,28.6,71.4,53.1,75.5,37.5,38.6,23.9,...,9.7,90.3,29.5,70.5,49.6,50.4,None,None,None,True
5024,2026-12-01 20:00:00+00:00,52.0,21.8,26.2,73.8,48.0,78.2,42.9,31.4,25.7,...,19.8,80.2,39.2,60.8,58.3,41.7,None,None,None,False
5025,2026-12-01 20:30:00+00:00,28.8,34.9,36.3,63.7,71.2,65.1,21.5,46.7,31.8,...,38.6,61.4,67.0,33.0,83.4,16.6,None,None,None,False
5026,2026-12-01 20:45:00+00:00,67.4,22.5,10.1,89.9,32.6,77.5,44.5,38.8,16.7,...,22.3,77.7,40.1,59.9,63.9,36.1,None,None,None,True


In [94]:
df.sort_values(by='time', ascending=False)

,time,chance1x2_chance_p1,chance1x2_chance_px,chance1x2_chance_p2,chance1x2_chance_p1x,chance1x2_chance_p2x,chance1x2_chance_p12,chance1x2_chance_pHt1,chance1x2_chance_pHtx,chance1x2_chance_pHt2,...,underOver_flashback_under05HT,underOver_flashback_over05HT,underOver_flashback_under15,underOver_flashback_over15,underOver_flashback_under25,underOver_flashback_over25,underOver_flashback_under35,underOver_flashback_over35,win_1,white_noise
5027,2026-12-01 21:00:00+00:00,30.0,28.7,41.3,58.7,70.0,71.3,25.4,46.5,28.1,...,34.5,65.5,33.3,66.7,58.9,41.1,79.6,20.4,0,8309.760162
5026,2026-12-01 20:45:00+00:00,67.4,22.5,10.1,89.9,32.6,77.5,44.5,38.8,16.7,...,21.9,78.1,22.3,77.7,40.1,59.9,63.9,36.1,1,12451.850863
5025,2026-12-01 20:30:00+00:00,28.8,34.9,36.3,63.7,71.2,65.1,21.5,46.7,31.8,...,39.2,60.8,38.6,61.4,67.0,33.0,83.4,16.6,0,16814.559481
5024,2026-12-01 20:00:00+00:00,52.0,21.8,26.2,73.8,48.0,78.2,42.9,31.4,25.7,...,23.6,76.4,19.8,80.2,39.2,60.8,58.3,41.7,0,10246.665352
5023,2026-12-01 18:30:00+00:00,46.9,24.5,28.6,71.4,53.1,75.5,37.5,38.6,23.9,...,17.8,82.2,9.7,90.3,29.5,70.5,49.6,50.4,1,22742.360514
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,2025-01-10 18:00:00+00:00,27.2,34.0,38.8,61.2,72.8,66.0,27.1,37.6,35.3,...,28.7,71.3,23.8,76.2,46.3,53.7,69.3,30.7,0,9005.144253
3,2025-01-10 02:35:00+00:00,37.9,33.6,28.5,71.5,62.1,66.4,24.4,52.2,23.4,...,37.6,62.4,35.4,64.6,62.1,37.9,83.3,16.7,0,16633.260204
2,2025-01-10 02:30:00+00:00,42.2,31.1,26.7,73.3,57.8,68.9,30.8,39.5,29.7,...,29.5,70.5,23.8,76.2,46.2,53.8,71.0,29.0,1,6368.008142
1,2025-01-10 02:30:00+00:00,53.8,26.0,20.2,79.8,46.2,74.0,43.9,40.8,15.3,...,35.5,64.5,33.6,66.4,59.0,41.0,79.5,20.5,0,13035.984277
